# Direct Gold Cross Crypto Macro Features 1d

Heavy notebook for the phase-1 cross-domain Gold feature product.

Responsibilities:

- build the daily market feature skeleton from Gold market returns and volatility tables
- align selected macro indicators from Gold macro to each `feature_date` using observation-date as-of fill
- pack macro values into `macro_features MAP<STRING, DOUBLE>`
- merge the result into `gld_cross.dp_crypto_macro_features_1d`
- write real-time pipeline observability records to `obs.obs_pipeline_run_log` and `obs.obs_ingestion_state`

Phase-1 fill contract:

- `fill_policy = asof_observation_date_ffill_v1`
- suitable for analytics and dashboarding
- not declared backtest-safe for macro release timing or revision availability


In [ ]:
import json
import uuid
from datetime import datetime, timedelta, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType
from pyspark.sql.window import Window

UTC = timezone.utc
DEFAULT_LOOKBACK_DAYS = 120
FILL_POLICY = "asof_observation_date_ffill_v1"
PIPELINE_NAME = "gold_cross_crypto_macro_features_1d"
SOURCE_NAME = "cross"

RUN_LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("source_name", StringType(), True),
    StructField("target_table", StringType(), False),
    StructField("status", StringType(), False),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("started_at", TimestampType(), False),
    StructField("completed_at", TimestampType(), True),
    StructField("error_message", StringType(), True),
    StructField("metadata_json", StringType(), True),
])

INGESTION_STATE_SCHEMA = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("source_name", StringType(), True),
    StructField("target_table", StringType(), False),
    StructField("watermark_value", StringType(), True),
    StructField("watermark_type", StringType(), True),
    StructField("last_success_at", TimestampType(), True),
    StructField("last_run_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("updated_at", TimestampType(), False),
])


def ensure_table_exists(table_name: str):
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table {table_name} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )


def build_metadata_json(payload: dict) -> str:
    return json.dumps(payload, sort_keys=True, default=str)


def parse_csv_values(raw_value: str, label: str) -> list[str]:
    values = []
    seen = set()
    for token in raw_value.split(","):
        normalized = token.strip().upper()
        if not normalized or normalized in seen:
            continue
        values.append(normalized)
        seen.add(normalized)

    if not values:
        raise ValueError(f"{label} must contain at least one comma-separated value")
    return values


def resolve_date_window(mode: str, start_date_raw: str, end_date_raw: str, lookback_days_raw: str):
    normalized_mode = mode.strip().lower()
    if normalized_mode not in {"incremental", "backfill"}:
        raise ValueError("mode must be either incremental or backfill")

    if normalized_mode == "backfill":
        if not start_date_raw or not end_date_raw:
            raise ValueError("start_date and end_date are required for backfill mode")
        start_date = datetime.strptime(start_date_raw, "%Y-%m-%d").date()
        end_date = datetime.strptime(end_date_raw, "%Y-%m-%d").date()
    else:
        latest_complete_date = datetime.now(UTC).date() - timedelta(days=1)
        lookback_days = int(lookback_days_raw or str(DEFAULT_LOOKBACK_DAYS))
        if lookback_days <= 0:
            raise ValueError("lookback_days must be a positive integer")
        end_date = latest_complete_date
        start_date = end_date - timedelta(days=lookback_days - 1)

    if start_date > end_date:
        raise ValueError("start_date must be less than or equal to end_date")

    return start_date, end_date


def upsert_pipeline_run_log(log_table: str, payload: dict):
    log_df = spark.createDataFrame([
        (
            payload["run_id"],
            payload["pipeline_name"],
            payload["layer"],
            payload.get("source_name"),
            payload["target_table"],
            payload["status"],
            payload.get("rows_read"),
            payload.get("rows_written"),
            payload["started_at"],
            payload.get("completed_at"),
            payload.get("error_message"),
            payload.get("metadata_json"),
        )
    ], schema=RUN_LOG_SCHEMA)

    DeltaTable.forName(spark, log_table).alias("tgt").merge(
        log_df.alias("src"),
        "tgt.run_id = src.run_id AND tgt.pipeline_name = src.pipeline_name AND tgt.target_table = src.target_table",
    ).whenMatchedUpdate(
        set={
            "layer": "src.layer",
            "source_name": "src.source_name",
            "status": "src.status",
            "rows_read": "src.rows_read",
            "rows_written": "src.rows_written",
            "started_at": "src.started_at",
            "completed_at": "src.completed_at",
            "error_message": "src.error_message",
            "metadata_json": "src.metadata_json",
        }
    ).whenNotMatchedInsertAll().execute()


def read_table_watermark(table_name: str, watermark_column: str):
    row = spark.table(table_name).agg(F.max(F.col(watermark_column)).alias("max_watermark")).collect()[0]
    max_watermark = row["max_watermark"]
    if max_watermark is None:
        return None
    if hasattr(max_watermark, "isoformat"):
        return max_watermark.isoformat()
    return str(max_watermark)


def upsert_ingestion_state(state_table: str, payload: dict):
    state_df = spark.createDataFrame([
        (
            payload["pipeline_name"],
            payload.get("source_name"),
            payload["target_table"],
            payload.get("watermark_value"),
            payload.get("watermark_type"),
            payload.get("last_success_at"),
            payload.get("last_run_id"),
            payload.get("status"),
            payload["updated_at"],
        )
    ], schema=INGESTION_STATE_SCHEMA)

    DeltaTable.forName(spark, state_table).alias("tgt").merge(
        state_df.alias("src"),
        "tgt.pipeline_name = src.pipeline_name AND tgt.target_table = src.target_table",
    ).whenMatchedUpdate(
        set={
            "source_name": "src.source_name",
            "watermark_value": "src.watermark_value",
            "watermark_type": "src.watermark_type",
            "last_success_at": "src.last_success_at",
            "last_run_id": "src.last_run_id",
            "status": "src.status",
            "updated_at": "src.updated_at",
        }
    ).whenNotMatchedInsertAll().execute()


dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("product_ids", "BTC-USD,ETH-USD,SOL-USD,LINK-USD,DOGE-USD")
dbutils.widgets.text("macro_indicator_ids", "ECB_FX_REF_EUR_USD,FRED_CPIAUCSL,FRED_FEDFUNDS,FRED_GDP")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", str(DEFAULT_LOOKBACK_DAYS))
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
catalog = dbutils.widgets.get("catalog").strip() or "market_macro"
raw_product_ids = dbutils.widgets.get("product_ids").strip()
raw_macro_indicator_ids = dbutils.widgets.get("macro_indicator_ids").strip()
mode = dbutils.widgets.get("mode").strip().lower()
start_date_raw = dbutils.widgets.get("start_date").strip()
end_date_raw = dbutils.widgets.get("end_date").strip()
lookback_days_raw = dbutils.widgets.get("lookback_days").strip()
run_id = dbutils.widgets.get("run_id").strip()

started_at = datetime.now(UTC)
returns_table = f"{catalog}.gld_market.dp_crypto_returns_1d"
volatility_table = f"{catalog}.gld_market.dp_crypto_volatility_1d"
macro_table = f"{catalog}.gld_macro.dp_macro_indicators"
target_table = f"{catalog}.gld_cross.dp_crypto_macro_features_1d"
state_table = f"{catalog}.obs.obs_ingestion_state"
run_log_table = f"{catalog}.obs.obs_pipeline_run_log"

ensure_table_exists(returns_table)
ensure_table_exists(volatility_table)
ensure_table_exists(macro_table)
ensure_table_exists(target_table)
ensure_table_exists(state_table)
ensure_table_exists(run_log_table)

product_ids = parse_csv_values(raw_product_ids, "product_ids")
macro_indicator_ids = parse_csv_values(raw_macro_indicator_ids, "macro_indicator_ids")
start_date, end_date = resolve_date_window(mode, start_date_raw, end_date_raw, lookback_days_raw)

upsert_pipeline_run_log(
    run_log_table,
    {
        "run_id": run_id,
        "pipeline_name": PIPELINE_NAME,
        "layer": "gold",
        "source_name": SOURCE_NAME,
        "target_table": target_table,
        "status": "started",
        "rows_read": None,
        "rows_written": None,
        "started_at": started_at,
        "completed_at": None,
        "error_message": None,
        "metadata_json": build_metadata_json(
            {
                "mode": mode,
                "product_ids": product_ids,
                "macro_indicator_ids": macro_indicator_ids,
                "start_date": start_date.isoformat(),
                "end_date": end_date.isoformat(),
                "fill_policy": FILL_POLICY,
                "backtest_safe": False,
                "returns_table": returns_table,
                "volatility_table": volatility_table,
                "macro_table": macro_table,
            }
        ),
    },
)

try:
    market_returns_df = (
        spark.table(returns_table)
        .select(
            "product_id",
            F.col("bar_date").alias("feature_date"),
            "base_asset",
            "quote_currency",
            "simple_return_1d",
            "log_return_1d",
        )
        .filter(F.col("product_id").isin(product_ids))
        .filter((F.col("feature_date") >= F.lit(start_date)) & (F.col("feature_date") <= F.lit(end_date)))
    )
    rows_market_returns_read = market_returns_df.count()

    market_volatility_df = (
        spark.table(volatility_table)
        .select(
            "product_id",
            F.col("bar_date").alias("feature_date"),
            "volatility_7d",
            "volatility_30d",
        )
        .filter(F.col("product_id").isin(product_ids))
        .filter((F.col("feature_date") >= F.lit(start_date)) & (F.col("feature_date") <= F.lit(end_date)))
    )
    rows_market_volatility_read = market_volatility_df.count()

    rows_read = rows_market_returns_read + rows_market_volatility_read

    if rows_market_returns_read == 0:
        result = {
            "status": "success_empty",
            "pipeline_name": PIPELINE_NAME,
            "catalog": catalog,
            "run_id": run_id,
            "product_ids": product_ids,
            "macro_indicator_ids": macro_indicator_ids,
            "mode": mode,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "source_market_returns_table": returns_table,
            "source_market_volatility_table": volatility_table,
            "source_macro_table": macro_table,
            "target_table": target_table,
            "fill_policy": FILL_POLICY,
            "backtest_safe": False,
            "rows_market_returns_read": rows_market_returns_read,
            "rows_market_volatility_read": rows_market_volatility_read,
            "rows_market_base": 0,
            "rows_macro_selected": 0,
            "rows_macro_asof_ready": 0,
            "rows_ready": 0,
            "rows_to_update": 0,
            "rows_to_insert": 0,
            "rows_merged": 0,
        }
        completed_at = datetime.now(UTC)
        upsert_pipeline_run_log(
            run_log_table,
            {
                "run_id": run_id,
                "pipeline_name": PIPELINE_NAME,
                "layer": "gold",
                "source_name": SOURCE_NAME,
                "target_table": target_table,
                "status": result["status"],
                "rows_read": rows_read,
                "rows_written": 0,
                "started_at": started_at,
                "completed_at": completed_at,
                "error_message": None,
                "metadata_json": build_metadata_json(result),
            },
        )
        upsert_ingestion_state(
            state_table,
            {
                "pipeline_name": PIPELINE_NAME,
                "source_name": SOURCE_NAME,
                "target_table": target_table,
                "watermark_value": read_table_watermark(target_table, "feature_date"),
                "watermark_type": "feature_date",
                "last_success_at": completed_at,
                "last_run_id": run_id,
                "status": result["status"],
                "updated_at": completed_at,
            },
        )
        result
    else:
        market_base_df = (
            market_returns_df.alias("ret")
            .join(
                market_volatility_df.alias("vol"),
                on=["product_id", "feature_date"],
                how="left",
            )
            .select(
                "product_id",
                "feature_date",
                "base_asset",
                "quote_currency",
                "simple_return_1d",
                "log_return_1d",
                "volatility_7d",
                "volatility_30d",
            )
        )
        rows_market_base = market_base_df.count()

        macro_source_df = (
            spark.table(macro_table)
            .select("indicator_id", "observation_date", "value", "computed_at", "run_id")
            .filter(F.col("indicator_id").isin(macro_indicator_ids))
            .filter(F.col("observation_date") <= F.lit(end_date))
        )
        rows_macro_selected = macro_source_df.count()
        rows_read += rows_macro_selected

        available_macro_indicator_ids = [
            row["indicator_id"]
            for row in macro_source_df.select("indicator_id").distinct().collect()
        ]
        missing_macro_indicator_ids = [
            indicator_id
            for indicator_id in macro_indicator_ids
            if indicator_id not in available_macro_indicator_ids
        ]
        if missing_macro_indicator_ids:
            raise RuntimeError(
                "Missing requested macro indicators in gld_macro.dp_macro_indicators: "
                + ", ".join(missing_macro_indicator_ids)
            )

        feature_dates_df = market_base_df.select("feature_date").distinct()
        indicator_ids_df = spark.createDataFrame([(indicator_id,) for indicator_id in macro_indicator_ids], ["indicator_id"])

        macro_asof_candidates_df = (
            feature_dates_df.crossJoin(indicator_ids_df)
            .join(
                macro_source_df,
                on="indicator_id",
                how="left",
            )
            .filter(F.col("observation_date").isNull() | (F.col("observation_date") <= F.col("feature_date")))
        )

        asof_window = Window.partitionBy("feature_date", "indicator_id").orderBy(
            F.col("observation_date").desc(),
            F.col("computed_at").desc(),
            F.col("run_id").desc(),
        )

        macro_asof_df = (
            macro_asof_candidates_df
            .withColumn("_row_number", F.row_number().over(asof_window))
            .filter((F.col("_row_number") == 1) & F.col("value").isNotNull())
            .select("feature_date", "indicator_id", "value")
        )
        rows_macro_asof_ready = macro_asof_df.count()

        macro_features_df = macro_asof_df.groupBy("feature_date").agg(
            F.map_from_entries(F.collect_list(F.struct(F.col("indicator_id"), F.col("value")))).alias("macro_features")
        )

        computed_at = datetime.now(UTC)
        final_df = (
            market_base_df
            .join(macro_features_df, on="feature_date", how="left")
            .withColumn("fill_policy", F.lit(FILL_POLICY))
            .withColumn("computed_at", F.lit(computed_at))
            .withColumn("run_id", F.lit(run_id))
            .select(
                "feature_date",
                "product_id",
                "base_asset",
                "quote_currency",
                "simple_return_1d",
                "log_return_1d",
                "volatility_7d",
                "volatility_30d",
                "macro_features",
                "fill_policy",
                "computed_at",
                "run_id",
            )
        )

        rows_ready = final_df.count()

        existing_key_count = (
            final_df.select("feature_date", "product_id")
            .join(
                spark.table(target_table).select("feature_date", "product_id"),
                on=["feature_date", "product_id"],
                how="inner",
            )
            .count()
            if rows_ready > 0
            else 0
        )

        if rows_ready > 0:
            DeltaTable.forName(spark, target_table).alias("tgt").merge(
                final_df.alias("src"),
                "tgt.feature_date = src.feature_date AND tgt.product_id = src.product_id",
            ).whenMatchedUpdate(
                set={
                    "base_asset": "src.base_asset",
                    "quote_currency": "src.quote_currency",
                    "simple_return_1d": "src.simple_return_1d",
                    "log_return_1d": "src.log_return_1d",
                    "volatility_7d": "src.volatility_7d",
                    "volatility_30d": "src.volatility_30d",
                    "macro_features": "src.macro_features",
                    "fill_policy": "src.fill_policy",
                    "computed_at": "src.computed_at",
                    "run_id": "src.run_id",
                }
            ).whenNotMatchedInsertAll().execute()

        display(final_df.orderBy("product_id", "feature_date"))

        result = {
            "status": "success",
            "pipeline_name": PIPELINE_NAME,
            "catalog": catalog,
            "run_id": run_id,
            "product_ids": product_ids,
            "macro_indicator_ids": macro_indicator_ids,
            "mode": mode,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "source_market_returns_table": returns_table,
            "source_market_volatility_table": volatility_table,
            "source_macro_table": macro_table,
            "target_table": target_table,
            "fill_policy": FILL_POLICY,
            "backtest_safe": False,
            "rows_market_returns_read": rows_market_returns_read,
            "rows_market_volatility_read": rows_market_volatility_read,
            "rows_market_base": rows_market_base,
            "rows_macro_selected": rows_macro_selected,
            "rows_macro_asof_ready": rows_macro_asof_ready,
            "rows_ready": rows_ready,
            "rows_to_update": existing_key_count,
            "rows_to_insert": rows_ready - existing_key_count,
            "rows_merged": rows_ready,
        }

        completed_at = datetime.now(UTC)
        upsert_pipeline_run_log(
            run_log_table,
            {
                "run_id": run_id,
                "pipeline_name": PIPELINE_NAME,
                "layer": "gold",
                "source_name": SOURCE_NAME,
                "target_table": target_table,
                "status": result["status"],
                "rows_read": rows_read,
                "rows_written": rows_ready,
                "started_at": started_at,
                "completed_at": completed_at,
                "error_message": None,
                "metadata_json": build_metadata_json(result),
            },
        )
        upsert_ingestion_state(
            state_table,
            {
                "pipeline_name": PIPELINE_NAME,
                "source_name": SOURCE_NAME,
                "target_table": target_table,
                "watermark_value": read_table_watermark(target_table, "feature_date"),
                "watermark_type": "feature_date",
                "last_success_at": completed_at,
                "last_run_id": run_id,
                "status": result["status"],
                "updated_at": completed_at,
            },
        )
        result
except Exception as exc:
    completed_at = datetime.now(UTC)
    upsert_pipeline_run_log(
        run_log_table,
        {
            "run_id": run_id,
            "pipeline_name": PIPELINE_NAME,
            "layer": "gold",
            "source_name": SOURCE_NAME,
            "target_table": target_table,
            "status": "failed",
            "rows_read": None,
            "rows_written": None,
            "started_at": started_at,
            "completed_at": completed_at,
            "error_message": str(exc),
            "metadata_json": build_metadata_json(
                {
                    "mode": mode,
                    "product_ids": product_ids,
                    "macro_indicator_ids": macro_indicator_ids,
                    "start_date": start_date.isoformat(),
                    "end_date": end_date.isoformat(),
                    "fill_policy": FILL_POLICY,
                }
            ),
        },
    )
    raise

result
